# Study Phase 2: data and frozen preprocessing

Run from the project environment. This rebuilds fresh paired representations
without legacy caches, fits preprocessing on original training only, and saves
an isolated readiness bundle. It loads held-out inputs for automatic validity
checks, but fits no detectors and computes no validation/test metrics or EDA.

In [1]:
import sys

import numpy as np
import pandas as pd

from cheatdetect.utils import _find_project_root

sys.path.insert(0, str(_find_project_root()))

from cheatdetect.data.dataset import ARTIFACT_ROOT, prepare_study, save_prepared

prepared = prepare_study()
directory = ARTIFACT_ROOT / "phase_02_readiness"
save_prepared(prepared, directory)

## Automated readiness summary

Each parent has one aligned nine-timestep sequence. Synthetic copy indices
identify copies, not additional independent sessions. Both augmentation states
reuse the same preprocessing objects and the same held-out representations.
Zero-window files remain in the frozen split and are explicitly reported below;
they contribute no windows. An entirely unusable required partition fails.

In [2]:
session_summary = prepared.session_report.groupby("partition", sort=False).agg(
    assigned_session_files=("session_file", "size"),
    contributing_session_files=("included", "sum"),
    parent_rows=("parent_windows", "sum"),
)
session_summary["excluded_session_files"] = (
    session_summary["assigned_session_files"] - session_summary["contributing_session_files"]
)
session_summary

,assigned_session_files,contributing_session_files,parent_rows,excluded_session_files
partition,,,,
normal_train,15,15,662,0
normal_val,4,4,197,0
mixed_val,5,5,196,0
mixed_test,12,10,366,2


In [3]:
prepared.session_report.loc[~prepared.session_report["included"]]

,partition,session_file,raw_rows,cleaned_events,parent_windows,included,exclusion_reason
26,mixed_test,mixed - El Farouk Omar.csv,32,0,0,False,no_cleaned_events
34,mixed_test,youssef ahmed- mixed.csv,32,0,0,False,no_cleaned_events


In [4]:
pd.DataFrame([
    {
        "partition": name,
        "parent_rows": len(view.parent),
        "sequence_rows": len(view.sequences),
        "contributing_original_session_files": view.identities["session_file"].nunique(),
        "nonfinite_raw_cells": int((~np.isfinite(view.parent.to_numpy())).sum()),
    }
    for name, view in prepared.raw.items()
])

,partition,parent_rows,sequence_rows,contributing_original_session_files,nonfinite_raw_cells
0,train_original,662,662,15,0
1,train_synthetic,1324,1324,15,0
2,normal_val,197,197,4,0
3,mixed_val,196,196,5,0
4,mixed_test,366,366,10,0


In [5]:
pd.DataFrame([
    {
        "recipe": name,
        "original_fit_rows": processor.n_original_rows_,
        "train_off_rows": len(prepared.training(name, False)),
        "train_on_rows": len(prepared.training(name, True)),
        "encoded_columns": len(processor.output_features_),
        "gamma_center": prepared.gamma_centers.get(name),
    }
    for name, processor in prepared.preprocessors.items()
])

,recipe,original_fit_rows,train_off_rows,train_on_rows,encoded_columns,gamma_center
0,IF-raw,662,662,1986,33,NaN
1,IF-log,662,662,1986,33,NaN
2,SVM-base,662,662,1986,33,0.043590
3,SVM-log,662,662,1986,33,0.045288
4,SVM-YJ,662,662,1986,33,0.045288
5,SVM-quantile,662,662,1986,33,0.045288
6,AE-base,5958,662,1986,33,NaN
7,AE-log,5958,662,1986,33,NaN
8,AE-YJ,5958,662,1986,33,NaN


## Reuse contract

Later study runs may load this trusted local bundle only with a freshly computed
`input_manifest` matching raw-file hashes, ordered splits, extraction/recipe
code, geometry, augmentation settings, and library versions. No artifact under
`dataset/processed/` or `best_models/` is overwritten.